# Módulo 1 · 01 — Datos y construcción de las series

**Objetivo:** descargar el conjunto *Operación de pasajeros y despacho de vehículos en la modalidad de transporte de pasajeros por carretera* (Superintendencia de Transporte, datos.gov.co), documentarlo y construir la serie objetivo **pasajeros por ruta y por día**.

Pendiente en este notebook:
1. Caso de uso (quién usa el pronóstico, qué decisión, con 30 días de anticipación).
2. Descarga y descripción: nº de registros, variables, periodo, rutas. Citar diccionario e inconsistencias.
3. Codificación DIVIPOLA (DANE, fuente oficial) para origen/destino.
4. Agregación diaria, días sin registros, anomalías (p. ej. despachos con 0 pasajeros).

In [ ]:
# parameters
DEMO = False
LIMITE_FILAS = 50_000  # paginación de la API Socrata

In [ ]:
import sys
from pathlib import Path
RAIZ = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(RAIZ))

from common.seeds import fijar_semilla
from common.config import cargar_config
fijar_semilla()

import pandas as pd
import requests

cfg = cargar_config("modulo1")
DATA = RAIZ / "modulo1_demanda" / "data"

## 1. Caso de uso

_TODO: redactar._

## 2. Descarga de datos

Se usa la API Socrata de datos.gov.co. El crudo se guarda en `data/raw/` (ignorado por git); la versión procesada en `data/processed/` sí se versiona.

In [ ]:
def descargar_socrata(dataset_id: str, limite: int = LIMITE_FILAS, max_paginas: int | None = None) -> pd.DataFrame:
    """Descarga paginada desde datos.gov.co."""
    url = f"{cfg['datos']['url_base']}/{dataset_id}.json"
    paginas, offset = [], 0
    while True:
        r = requests.get(url, params={"$limit": limite, "$offset": offset}, timeout=120)
        r.raise_for_status()
        lote = r.json()
        if not lote:
            break
        paginas.append(pd.DataFrame(lote))
        offset += limite
        if max_paginas and len(paginas) >= max_paginas:
            break
    return pd.concat(paginas, ignore_index=True)

ruta_raw = DATA / "raw" / "despachos.parquet"
# TODO: fijar cfg["datos"]["dataset_id"] y ejecutar
# if not ruta_raw.exists():
#     df_raw = descargar_socrata(cfg["datos"]["dataset_id"], max_paginas=2 if DEMO else None)
#     df_raw.to_parquet(ruta_raw)
# df_raw = pd.read_parquet(ruta_raw)

## 3. Descripción del conjunto y diccionario

_TODO: registros, variables, periodo, rutas; significado de variables citando el diccionario; inconsistencias._

## 4. DIVIPOLA

_TODO: descargar la codificación oficial del DANE a `data/external/` y citarla._

## 5. Construcción de la serie pasajeros/ruta/día

In [ ]:
def construir_series(df: pd.DataFrame, col_fecha: str, col_ruta: str, col_pasajeros: str) -> pd.DataFrame:
    """Agrega a pasajeros por ruta y día y completa el calendario (días sin registro explícitos).

    TODO: decidir tratamiento de días sin registros (0 vs NaN) y de anomalías.
    """
    serie = (df.assign(fecha=pd.to_datetime(df[col_fecha]).dt.normalize())
               .groupby([col_ruta, "fecha"])[col_pasajeros].sum()
               .unstack(col_ruta))
    calendario = pd.date_range(serie.index.min(), serie.index.max(), freq="D")
    return serie.reindex(calendario)

## 6. Exportar datos procesados

`data/processed/series_diarias.parquet` es la entrada de los notebooks 02–04.